In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SILVER_SCHEMA = "bakehouse_silver"
GOLD_SCHEMA = "bakehouse_gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

txn = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_transactions")
customers = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_customers")
franchises = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_franchises")

##Customer segments (RFM)

In [0]:
max_date = txn.agg(F.max("dateTime")).collect()[0][0]

rfm_base = (
    txn.groupBy("customerID")
    .agg(
        F.datediff(F.lit(max_date), F.max("dateTime")).alias("recency_days"),
        F.count("transactionID").alias("frequency"),
        F.sum("totalPrice").alias("monetary")
    )
)

# quartile scoring: 4 = best, 1 = worst, in each dimension
rfm_scored = (
    rfm_base
    .withColumn("recency_score", F.ntile(4).over(Window.orderBy(F.col("recency_days").desc()))) 
    .withColumn("frequency_score", F.ntile(4).over(Window.orderBy("frequency")))
    .withColumn("monetary_score", F.ntile(4).over(Window.orderBy("monetary")))
)

customer_segments = (
    rfm_scored
    .withColumn(
        "segment",
        F.when((F.col("recency_score") >= 3) & (F.col("frequency_score") >= 3) & (F.col("monetary_score") >= 3), "Champions")
         .when((F.col("recency_score") <= 2) & (F.col("frequency_score") >= 3), "At Risk")
         .when((F.col("recency_score") <= 2) & (F.col("frequency_score") <= 2), "Lost")
         .otherwise("Regular")
    )
    .join(customers.select("customerID", "first_name", "last_name", "city", "state", "country"), "customerID", "left")
)

customer_segments.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_customer_segments")

print("Segment distribution:")
customer_segments.groupBy("segment").agg(
    F.count("*").alias("customers"),
    F.sum("monetary").alias("total_revenue")
).orderBy(F.desc("total_revenue")).show()

##Franchise performance

In [0]:
franchise_perf = (
    txn.groupBy("franchiseID")
    .agg(
        F.sum("totalPrice").alias("total_revenue"),
        F.count("transactionID").alias("total_transactions"),
        F.countDistinct("customerID").alias("unique_customers"),
        F.round(F.avg("totalPrice"), 2).alias("avg_transaction_value")
    )
    .join(franchises, "franchiseID", "left")
)

franchise_perf.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_franchise_performance")

print("Top 5 franchises by revenue:")
franchise_perf.orderBy(F.desc("total_revenue")).select(
    "franchiseID", "name", "city", "country", "total_revenue", "total_transactions", "unique_customers"
).show(5, truncate=False)

print("Bottom 5 franchises by revenue:")
franchise_perf.orderBy("total_revenue").select(
    "franchiseID", "name", "city", "country", "total_revenue", "total_transactions", "unique_customers"
).show(5, truncate=False)

##Geography rollup

In [0]:
geo_perf = (
    franchise_perf.groupBy("country")
    .agg(
        F.sum("total_revenue").alias("country_revenue"),
        F.countDistinct("franchiseID").alias("franchise_count"),
        F.sum("unique_customers").alias("total_customers")
    )
    .withColumn("revenue_per_franchise", F.round(F.col("country_revenue") / F.col("franchise_count"), 2))
    .orderBy(F.desc("revenue_per_franchise"))
)

geo_perf.show(truncate=False)